# MIRFLICKR Thorough EDA Add-on

Append these sections after your current MIRFLICKR notebook.  
It assumes these variables already exist:

- `spark`
- `F`, `T`
- `image_df`
- `features_df`
- `OUTPUT_ROOT`
- `save_current_chart`

This add-on includes:
1. dataset overview and schema
2. missingness / data quality
3. numeric summaries
4. file size, resolution, and aspect ratio analysis
5. EXIF and camera analysis
6. tag statistics and long-tail analysis
7. temporal analysis
8. location analysis
9. semantic bucket analysis
10. memory-candidate analysis
11. image grids for demo

In [ ]:
# =========================
# 1. BASIC OVERVIEW
# =========================

print("Row count:", features_df.count())
print("Column count:", len(features_df.columns))
print("Columns:")
for c in features_df.columns:
    print(" -", c)

features_df.printSchema()
display(features_df.limit(5))

In [ ]:
# =========================
# 2. DATA QUALITY / MISSINGNESS SUMMARY
# =========================

total_rows = features_df.count()

missing_exprs = [
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in features_df.columns
]

missing_pd = features_df.select(*missing_exprs).toPandas().T.reset_index()
missing_pd.columns = ["column", "null_count"]
missing_pd["null_pct"] = (missing_pd["null_count"] / total_rows * 100).round(2)
missing_pd = missing_pd.sort_values(["null_pct", "null_count"], ascending=False)

display(missing_pd.head(30))

In [ ]:
plt.figure(figsize=(12, 8))
top_missing = missing_pd.head(20).sort_values("null_pct", ascending=True)
plt.barh(top_missing["column"], top_missing["null_pct"])
plt.title("Top 20 Columns by Missing Percentage")
plt.xlabel("Null percentage")
plt.ylabel("Column")
save_current_chart("eda_missing_top20.png")
plt.show()

In [ ]:
# =========================
# 3. DUPLICATE CHECKS
# =========================

dup_pd = (
    features_df.groupBy("photo_id")
    .count()
    .where(F.col("count") > 1)
    .orderBy(F.desc("count"))
    .toPandas()
)

print("Duplicate photo_id rows:", len(dup_pd))
display(dup_pd.head(20))

In [ ]:
# =========================
# 4. NUMERIC SUMMARY
# =========================

numeric_cols = [
    "file_size_bytes", "width", "height", "aspect_ratio",
    "gps_lat", "gps_lon", "unique_tag_count", "tag_token_count",
    "exif_width", "exif_height"
]

existing_numeric_cols = [c for c in numeric_cols if c in features_df.columns]
summary_df = features_df.select(existing_numeric_cols).summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
)
display(summary_df)

In [ ]:
# =========================
# 5. FILE SIZE EDA
# =========================

file_pd = (
    features_df
    .select(
        (F.col("file_size_bytes") / 1024).alias("file_size_kb"),
        (F.col("file_size_bytes") / (1024 * 1024)).alias("file_size_mb")
    )
    .where(F.col("file_size_kb").isNotNull())
    .sample(False, 0.5, 42)
    .toPandas()
)

plt.figure(figsize=(10, 5))
plt.hist(file_pd["file_size_kb"], bins=60)
plt.title("File Size Distribution (KB)")
plt.xlabel("File size (KB)")
plt.ylabel("Frequency")
save_current_chart("eda_file_size_kb_hist.png")
plt.show()

plt.figure(figsize=(10, 5))
plt.boxplot(file_pd["file_size_mb"].dropna(), vert=False)
plt.title("File Size Boxplot (MB)")
plt.xlabel("MB")
save_current_chart("eda_file_size_mb_box.png")
plt.show()

In [ ]:
# =========================
# 6. RESOLUTION / MEGAPIXEL EDA
# =========================

res_df = (
    features_df
    .withColumn("megapixels", (F.col("width") * F.col("height")) / 1000000.0)
    .select("width", "height", "megapixels", "orientation", "aspect_ratio")
    .where(F.col("width").isNotNull() & F.col("height").isNotNull())
)

res_pd = res_df.sample(False, 0.5, 42).toPandas()

plt.figure(figsize=(7, 7))
plt.scatter(res_pd["width"], res_pd["height"], s=8, alpha=0.35)
plt.title("Image Resolution Scatter")
plt.xlabel("Width")
plt.ylabel("Height")
save_current_chart("eda_resolution_scatter.png")
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(res_pd["megapixels"].dropna(), bins=50)
plt.title("Megapixel Distribution")
plt.xlabel("Megapixels")
plt.ylabel("Frequency")
save_current_chart("eda_megapixels_hist.png")
plt.show()

In [ ]:
# =========================
# 7. ORIENTATION / ASPECT CATEGORY
# =========================

aspect_bucket_df = (
    features_df
    .withColumn(
        "aspect_bucket",
        F.when(F.col("aspect_ratio").isNull(), "unknown")
         .when(F.col("aspect_ratio") < 0.9, "tall")
         .when((F.col("aspect_ratio") >= 0.9) & (F.col("aspect_ratio") <= 1.1), "square-ish")
         .when((F.col("aspect_ratio") > 1.1) & (F.col("aspect_ratio") <= 1.8), "standard-wide")
         .otherwise("ultra-wide")
    )
)

aspect_pd = (
    aspect_bucket_df.groupBy("aspect_bucket")
    .count()
    .orderBy(F.desc("count"))
    .toPandas()
)

plt.figure(figsize=(8, 4))
plt.bar(aspect_pd["aspect_bucket"], aspect_pd["count"])
plt.title("Aspect Ratio Categories")
plt.xlabel("Aspect bucket")
plt.ylabel("Count")
save_current_chart("eda_aspect_bucket.png")
plt.show()

In [ ]:
# =========================
# 8. EXIF COVERAGE DETAIL
# =========================

exif_cov = features_df.select(
    F.sum(F.col("capture_ts").isNotNull().cast("int")).alias("capture_ts"),
    F.sum(F.col("camera_make").isNotNull().cast("int")).alias("camera_make"),
    F.sum(F.col("camera_model").isNotNull().cast("int")).alias("camera_model"),
    F.sum(F.col("gps_lat").isNotNull().cast("int")).alias("gps_lat"),
    F.sum(F.col("gps_lon").isNotNull().cast("int")).alias("gps_lon"),
    F.sum(F.col("iso").isNotNull().cast("int")).alias("iso"),
    F.sum(F.col("exposure_time").isNotNull().cast("int")).alias("exposure_time"),
    F.sum(F.col("f_number").isNotNull().cast("int")).alias("f_number"),
    F.sum(F.col("focal_length").isNotNull().cast("int")).alias("focal_length")
).toPandas().T.reset_index()

exif_cov.columns = ["field", "count"]
display(exif_cov)

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(exif_cov["field"], exif_cov["count"])
plt.title("EXIF Field Coverage")
plt.xlabel("EXIF field")
plt.ylabel("Count")
plt.xticks(rotation=30)
save_current_chart("eda_exif_field_coverage.png")
plt.show()

In [ ]:
# =========================
# 9. CAMERA ANALYSIS
# =========================

camera_make_pd = (
    features_df.where(F.col("camera_make").isNotNull())
    .groupBy("camera_make")
    .count()
    .orderBy(F.desc("count"))
    .limit(15)
    .toPandas()
)

camera_model_pd = (
    features_df.where(F.col("camera_model").isNotNull())
    .groupBy("camera_model")
    .count()
    .orderBy(F.desc("count"))
    .limit(15)
    .toPandas()
)

if len(camera_make_pd) > 0:
    plt.figure(figsize=(10, 6))
    plt.barh(camera_make_pd["camera_make"][::-1], camera_make_pd["count"][::-1])
    plt.title("Top Camera Makes")
    plt.xlabel("Count")
    save_current_chart("eda_top_camera_makes.png")
    plt.show()

if len(camera_model_pd) > 0:
    plt.figure(figsize=(10, 6))
    plt.barh(camera_model_pd["camera_model"][::-1], camera_model_pd["count"][::-1])
    plt.title("Top Camera Models")
    plt.xlabel("Count")
    save_current_chart("eda_top_camera_models.png")
    plt.show()

In [ ]:
# =========================
# 10. TAG COVERAGE AND LONG-TAIL
# =========================

tag_overview_pd = features_df.select(
    F.count("*").alias("n_images"),
    F.sum(F.col("has_tags").cast("int")).alias("with_tags"),
    F.avg(F.col("unique_tag_count")).alias("avg_unique_tag_count"),
    F.expr("percentile_approx(unique_tag_count, 0.5)").alias("median_unique_tag_count"),
    F.max("unique_tag_count").alias("max_unique_tag_count")
).toPandas()

display(tag_overview_pd)

In [ ]:
tag_count_pd = (
    features_df.select("unique_tag_count")
    .where(F.col("unique_tag_count").isNotNull())
    .toPandas()
)

plt.figure(figsize=(10, 5))
plt.hist(tag_count_pd["unique_tag_count"], bins=40)
plt.title("Unique Tag Count per Image")
plt.xlabel("Unique tag count")
plt.ylabel("Frequency")
save_current_chart("eda_unique_tag_count_hist.png")
plt.show()

In [ ]:
top_tags_pd = (
    features_df
    .select(F.explode_outer("tags").alias("tag"))
    .where(F.col("tag").isNotNull())
    .groupBy("tag")
    .count()
    .orderBy(F.desc("count"))
    .limit(40)
    .toPandas()
)

plt.figure(figsize=(12, 10))
plt.barh(top_tags_pd["tag"][::-1], top_tags_pd["count"][::-1])
plt.title("Top 40 Tags")
plt.xlabel("Count")
plt.ylabel("Tag")
save_current_chart("eda_top_40_tags.png")
plt.show()

display(top_tags_pd.head(20))

In [ ]:
tag_freq_df = (
    features_df
    .select(F.explode_outer("tags").alias("tag"))
    .where(F.col("tag").isNotNull())
    .groupBy("tag")
    .count()
)

tail_pd = tag_freq_df.select(
    F.count("*").alias("n_unique_tags"),
    F.sum((F.col("count") == 1).cast("int")).alias("tags_once"),
    F.sum((F.col("count") <= 5).cast("int")).alias("tags_le_5"),
    F.sum((F.col("count") <= 10).cast("int")).alias("tags_le_10")
).toPandas()

display(tail_pd)

In [ ]:
# =========================
# 11. TAG CO-OCCURRENCE (TOP TAGS)
# =========================

selected_tags = [t for t in top_tags_pd["tag"].head(12).tolist()]

co_pd = (
    features_df
    .select("photo_id", F.explode_outer("tags").alias("tag"))
    .where(F.col("tag").isin(selected_tags))
    .toPandas()
)

import pandas as pd
mat = pd.DataFrame(0, index=selected_tags, columns=selected_tags)

pairs = co_pd.groupby("photo_id")["tag"].apply(list)
for arr in pairs:
    arr = sorted(set(arr))
    for i in range(len(arr)):
        for j in range(len(arr)):
            mat.loc[arr[i], arr[j]] += 1

plt.figure(figsize=(10, 8))
plt.imshow(mat.values)
plt.xticks(range(len(selected_tags)), selected_tags, rotation=90)
plt.yticks(range(len(selected_tags)), selected_tags)
plt.title("Tag Co-occurrence Matrix")
plt.colorbar()
save_current_chart("eda_tag_cooccurrence_matrix.png")
plt.show()

In [ ]:
# =========================
# 12. TEMPORAL EDA
# =========================

time_cov = features_df.select(
    F.sum(F.col("year").isNotNull().cast("int")).alias("with_year"),
    F.sum(F.col("month").isNotNull().cast("int")).alias("with_month"),
    F.sum(F.col("day").isNotNull().cast("int")).alias("with_day")
).toPandas()

display(time_cov)

In [ ]:
year_pd = (
    features_df.where(F.col("year").isNotNull())
    .groupBy("year")
    .count()
    .orderBy("year")
    .toPandas()
)

if len(year_pd) > 0:
    plt.figure(figsize=(11, 5))
    plt.plot(year_pd["year"], year_pd["count"], marker="o")
    plt.title("Image Count by Year")
    plt.xlabel("Year")
    plt.ylabel("Count")
    save_current_chart("eda_year_trend.png")
    plt.show()

month_pd = (
    features_df.where(F.col("month").isNotNull())
    .groupBy("month")
    .count()
    .orderBy("month")
    .toPandas()
)

if len(month_pd) > 0:
    plt.figure(figsize=(9, 4))
    plt.bar(month_pd["month"], month_pd["count"])
    plt.title("Image Count by Month")
    plt.xlabel("Month")
    plt.ylabel("Count")
    save_current_chart("eda_month_distribution.png")
    plt.show()

In [ ]:
time_detail_df = (
    features_df
    .where(F.col("capture_ts").isNotNull())
    .withColumn("dow", F.date_format("capture_ts", "E"))
    .withColumn("hour", F.hour("capture_ts"))
)

dow_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow_pd = time_detail_df.groupBy("dow").count().toPandas()

if len(dow_pd) > 0:
    import pandas as pd
    dow_pd["dow"] = pd.Categorical(dow_pd["dow"], categories=dow_order, ordered=True)
    dow_pd = dow_pd.sort_values("dow")

    plt.figure(figsize=(8, 4))
    plt.bar(dow_pd["dow"], dow_pd["count"])
    plt.title("Image Count by Day of Week")
    plt.xlabel("Day")
    plt.ylabel("Count")
    save_current_chart("eda_day_of_week.png")
    plt.show()

hour_pd = time_detail_df.groupBy("hour").count().orderBy("hour").toPandas()
if len(hour_pd) > 0:
    plt.figure(figsize=(10, 4))
    plt.bar(hour_pd["hour"], hour_pd["count"])
    plt.title("Image Count by Hour of Day")
    plt.xlabel("Hour")
    plt.ylabel("Count")
    save_current_chart("eda_hour_distribution.png")
    plt.show()

In [ ]:
# =========================
# 13. LOCATION EDA
# =========================

location_cov_pd = features_df.select(
    F.sum(F.col("has_gps_exif").cast("int")).alias("with_gps_exif"),
    F.sum(F.col("has_location_tag").cast("int")).alias("with_location_tag"),
    F.sum((F.col("has_gps_exif") | F.col("has_location_tag")).cast("int")).alias("with_any_location_signal")
).toPandas()

display(location_cov_pd)

In [ ]:
top_locations_pd = (
    features_df
    .select(F.explode_outer("location_candidates").alias("location"))
    .where(F.col("location").isNotNull())
    .groupBy("location")
    .count()
    .orderBy(F.desc("count"))
    .limit(30)
    .toPandas()
)

plt.figure(figsize=(12, 9))
plt.barh(top_locations_pd["location"][::-1], top_locations_pd["count"][::-1])
plt.title("Top Extracted Locations from Tags")
plt.xlabel("Count")
plt.ylabel("Location")
save_current_chart("eda_top_locations.png")
plt.show()

display(top_locations_pd.head(20))

In [ ]:
loc_bucket_pd = (
    features_df
    .where(F.col("primary_location_tag").isNotNull())
    .groupBy("primary_location_tag", "semantic_bucket")
    .count()
    .orderBy(F.desc("count"))
    .limit(100)
    .toPandas()
)

display(loc_bucket_pd.head(20))

In [ ]:
# =========================
# 14. SEMANTIC BUCKET EDA
# =========================

bucket_pd = (
    features_df.groupBy("semantic_bucket")
    .count()
    .orderBy(F.desc("count"))
    .toPandas()
)

plt.figure(figsize=(10, 5))
plt.bar(bucket_pd["semantic_bucket"], bucket_pd["count"])
plt.title("Semantic Bucket Distribution")
plt.xlabel("Bucket")
plt.ylabel("Count")
plt.xticks(rotation=25)
save_current_chart("eda_semantic_buckets.png")
plt.show()

In [ ]:
bucket_loc_pd = (
    features_df
    .withColumn("location_signal", F.when(F.col("has_location_tag") | F.col("has_gps_exif"), "yes").otherwise("no"))
    .groupBy("semantic_bucket", "location_signal")
    .count()
    .toPandas()
)

pivot = bucket_loc_pd.pivot(index="semantic_bucket", columns="location_signal", values="count").fillna(0)
display(pivot)

In [ ]:
# =========================
# 15. MEMORY-CANDIDATE EDA
# =========================

story_eda_df = (
    features_df
    .withColumn(
        "memory_location",
        F.coalesce(
            F.col("primary_location_tag"),
            F.when(
                F.col("gps_lat").isNotNull() & F.col("gps_lon").isNotNull(),
                F.concat(F.lit("gps:"), F.round("gps_lat", 2).cast("string"), F.lit(","), F.round("gps_lon", 2).cast("string"))
            )
        )
    )
    .withColumn("memory_month", F.when(F.col("capture_ts").isNotNull(), F.date_format("capture_ts", "yyyy-MM")))
    .withColumn(
        "memory_candidate",
        F.col("memory_location").isNotNull() &
        F.col("memory_month").isNotNull() &
        F.col("semantic_bucket").isNotNull() &
        (F.col("semantic_bucket") != "unknown")
    )
)

memory_cov_pd = story_eda_df.select(
    F.count("*").alias("total"),
    F.sum(F.col("memory_candidate").cast("int")).alias("memory_candidates")
).toPandas()

display(memory_cov_pd)

In [ ]:
top_memory_clusters_pd = (
    story_eda_df
    .where(F.col("memory_candidate"))
    .groupBy("memory_month", "memory_location", "semantic_bucket")
    .count()
    .orderBy(F.desc("count"))
    .limit(30)
    .toPandas()
)

display(top_memory_clusters_pd.head(20))

if len(top_memory_clusters_pd) > 0:
    plt.figure(figsize=(12, 8))
    labels = [
        f"{r['memory_month']} | {r['memory_location']} | {r['semantic_bucket']}"
        for _, r in top_memory_clusters_pd.head(15).iterrows()
    ]
    vals = top_memory_clusters_pd.head(15)["count"].tolist()
    plt.barh(labels[::-1], vals[::-1])
    plt.title("Top Memory Clusters")
    plt.xlabel("Image count")
    plt.ylabel("Cluster")
    save_current_chart("eda_top_memory_clusters.png")
    plt.show()

In [ ]:
# =========================
# 16. IMAGE GRID HELPERS
# =========================

from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

def spark_file_uri_to_local_path(path_str):
    if path_str is None:
        return None
    if path_str.startswith("file:"):
        return path_str.replace("file:", "", 1)
    return path_str

def load_and_prepare_image(path_str, thumb_size=(120, 120)):
    try:
        local_path = spark_file_uri_to_local_path(path_str)
        with Image.open(local_path) as img:
            img = img.convert("RGB")
            img.thumbnail(thumb_size)
            canvas = Image.new("RGB", thumb_size, (245, 245, 245))
            x = (thumb_size[0] - img.size[0]) // 2
            y = (thumb_size[1] - img.size[1]) // 2
            canvas.paste(img, (x, y))
            return canvas
    except Exception:
        fallback = Image.new("RGB", thumb_size, (230, 230, 230))
        draw = ImageDraw.Draw(fallback)
        draw.text((10, 50), "ERR", fill=(80, 80, 80))
        return fallback

def show_image_grid(pdf, title, rows=10, cols=10, thumb_size=(120, 120)):
    n = min(len(pdf), rows * cols)
    fig, axes = plt.subplots(rows, cols, figsize=(18, 18))
    axes = axes.flatten()

    for i in range(rows * cols):
        ax = axes[i]
        ax.axis("off")
        if i < n:
            row = pdf.iloc[i]
            img = load_and_prepare_image(row["image_path"], thumb_size=thumb_size)
            ax.imshow(img)
            pid = row["photo_id"] if "photo_id" in pdf.columns else ""
            ax.set_title(f"id={pid}", fontsize=7)

    plt.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.show()

In [ ]:
# =========================
# 17. 10x10 LOCATION GRID
# =========================

TARGET_LOCATION = "London"

location_grid_pd = (
    features_df
    .where(F.array_contains(F.col("location_candidates"), TARGET_LOCATION))
    .orderBy(F.rand(42))
    .select("photo_id", "image_path", "primary_location_tag", "capture_ts")
    .limit(100)
    .toPandas()
)

print(f"Images found for {TARGET_LOCATION}: {len(location_grid_pd)}")
display(location_grid_pd.head())

if len(location_grid_pd) > 0:
    show_image_grid(location_grid_pd, f"10x10 Grid for Location: {TARGET_LOCATION}", rows=10, cols=10)

In [ ]:
# =========================
# 18. 10x10 SEMANTIC BUCKET GRID
# =========================

TARGET_BUCKET = "landscape_nature"

bucket_grid_pd = (
    features_df
    .where(F.col("semantic_bucket") == TARGET_BUCKET)
    .orderBy(F.rand(42))
    .select("photo_id", "image_path", "semantic_bucket", "capture_ts")
    .limit(100)
    .toPandas()
)

print(f"Images found for bucket {TARGET_BUCKET}: {len(bucket_grid_pd)}")
display(bucket_grid_pd.head())

if len(bucket_grid_pd) > 0:
    show_image_grid(bucket_grid_pd, f"10x10 Grid for Semantic Bucket: {TARGET_BUCKET}", rows=10, cols=10)

In [ ]:
# =========================
# 19. MEMORY-STYLE LOCATION + TIME GRID
# =========================

TARGET_LOCATION = "London"
TARGET_MONTH = "2008-07"

memory_grid_pd = (
    story_eda_df
    .where(F.col("memory_candidate"))
    .where(F.col("memory_location") == TARGET_LOCATION)
    .where(F.col("memory_month") == TARGET_MONTH)
    .orderBy(F.rand(42))
    .select("photo_id", "image_path", "memory_location", "memory_month", "semantic_bucket")
    .limit(100)
    .toPandas()
)

print(f"Images found for {TARGET_LOCATION} in {TARGET_MONTH}: {len(memory_grid_pd)}")
display(memory_grid_pd.head())

if len(memory_grid_pd) > 0:
    show_image_grid(memory_grid_pd, f"Memory-style Grid: {TARGET_LOCATION} | {TARGET_MONTH}", rows=10, cols=10)

In [ ]:
# =========================
# 20. OPTIONAL DUPLICATE CHECK BY FILE SIZE
# =========================

dup_size_pd = (
    features_df
    .groupBy("file_size_bytes")
    .count()
    .where(F.col("count") > 1)
    .orderBy(F.desc("count"))
    .limit(20)
    .toPandas()
)

display(dup_size_pd)

## Best charts to keep in your presentation

Use these:
1. metadata coverage
2. file size distribution
3. resolution scatter
4. top tags
5. top extracted locations
6. image count by year/month
7. semantic bucket distribution
8. top memory clusters
9. 10x10 location grid
10. 10x10 memory-style grid